# AIStylist — local virtual try-on demo

Runs **Qwen-Image-Edit** (open weights, Apache 2.0) inside ComfyUI on this notebook's GPU.
Once the models are downloaded, the generation server runs with **all outbound network blocked**,
so no external API (OpenAI or other) can be involved.

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100). *Settings → Internet*: **On** (only needed for step 2).
2. *Add Input → Upload*: create a **private** dataset with the customer photo and the product photos
   (and, optionally, the OpenAI outputs for the same sets). Name the files as in step 4 below.
3. *Run All*. The first run downloads about 22 GB and takes 15–25 minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the demo code

In [ ]:
import sys, subprocess
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "claude/dazzling-clarke-t0gw28",
                "https://github.com/Rashiin/style", "/tmp/style"], check=False)
sys.path.insert(0, "/tmp/style/demo")
import tryon_demo as td

## 2. Install ComfyUI and download the models (the only step that uses the internet)

In [ ]:
td.setup(version="2511")   # falls back to 2509 automatically if 2511 files are missing

## 3. Start the generator with the network blocked, and prove it

From here on the ComfyUI process cannot open any connection except to itself (localhost).
The next cell sends a request to `api.openai.com` under the same guard: it must fail.

In [ ]:
td.start_server(offline=True)
td.prove_offline()

## 4. Inputs

Put the files in your private Kaggle dataset. The cell finds them by name anywhere under `/kaggle/input`.
Use clean product photos from the product page (front view, plain background, no overlays like "5/8").

In [ ]:
from pathlib import Path

def find(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input")
    return hits[0]

def find_optional(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    return hits[0] if hits else None

PERSON = find("person.*")

# Keep the site logo (top-right corner of the customer photo) exactly as in the original.
LOGO_BOX = [(0.70, 0.0, 1.0, 0.085)]

KEEP = "Keep his face, hair, beard, skin tone, body pose, background and the AISTYLIST logo exactly the same. Photorealistic fashion photo."

SETS = {
    # Set 1: the client's own pick — blazer + loafers in one pass.
    "set1": [
        {"products": [find("blazer.*"), find("loafers.*")],
         "prompt": "Dress the man in picture 1 in the double-breasted pinstripe blazer from picture 2, buttoned, "
                   "worn over his striped shirt with the shirt collar visible, and replace his shoes with the "
                   "brown suede loafers with white soles from picture 3. Keep his trousers unchanged. " + KEEP},
    ],
    # Set 2 / set 3: add them when the product photos are in the dataset (uncomment and rename).
    # "set2": [{"products": [find("top.*"), find("trousers.*")],
    #           "prompt": "Dress the man in picture 1 in the top from picture 2 and the trousers from picture 3. " + KEEP}],
    # "set3": [{"products": [find("item.*"), find("accessory.*")],
    #           "prompt": "Dress the man in picture 1 in the garment from picture 2 and add the accessory from picture 3. " + KEEP}],
}

# If a set with several items does not come out well in one pass, split it into two passes, e.g.:
# SETS["set1"] = [
#     {"products": [find("blazer.*")], "prompt": "Dress the man in picture 1 in the double-breasted pinstripe blazer from picture 2, buttoned, over his striped shirt. " + KEEP},
#     {"products": [find("loafers.*")], "prompt": "Replace the shoes of the man in picture 1 with the brown suede loafers with white soles from picture 2. " + KEEP},
# ]

## 5. Generate

In [ ]:
from IPython.display import display, Image as IPImage

RESULTS = {}
for name, passes in SETS.items():
    RESULTS[name] = td.run_passes(PERSON, passes, seed=42, name=name, keep_regions=LOGO_BOX)
    display(IPImage(filename=str(RESULTS[name][-1].output), width=420))

## 6. Side-by-side comparison and report

If the OpenAI output for a set is in the dataset as `openai_set1.*`, `openai_set2.*`, … it is added as a fourth column.

In [ ]:
for name, passes in RESULTS.items():
    products = [p for step in SETS[name] for p in step["products"]]
    sheet = td.make_comparison(PERSON, products, passes[-1].output, openai=find_optional(f"openai_{name}.*"))
    display(IPImage(filename=str(sheet), width=1000))

td.write_report(RESULTS, offline=True)
!cd /kaggle/working && zip -qr tryon_output.zip tryon_output && ls -la /kaggle/working/tryon_output

## Troubleshooting

- **Out of GPU memory**: run `td.stop_server(); td.start_server(offline=True, extra_args=["--lowvram"])`, or generate at a smaller size: `td.run_passes(..., megapixels=0.75)`.
- **Black or noisy image**: try another `seed`, or split the set into two passes (see step 4).
- **Anything else**: `print(open("/tmp/tryon/comfyui.log").read()[-5000:])` shows the generator log.